# ME2N — Centro 4014 — Datalake

**Tabela:** `dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao`
**Domínio:** Pedidos de compra (view de exibicao)
**Filtro do cenário:** `cod_centro = '4014'`
**Colunas:** 47 · **Clustering declarado:** _(nenhum)_

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente com o filtro do centro embutido. Não há widget, view temporária nem ordem obrigatória.

## Objetivo

Extrair e caracterizar **toda** a base do centro 4014 para comparação com o extrato do SAP.

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados da tabela |
| 2 | Volumetria e representatividade do cenário |
| 3 | Confirmação do filtro |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| 6 | Preenchimento de todas as colunas |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 9 | Perfil numérico |
| **10** | **Totais para conciliação com o SAP** |
| 11 | Datas |
| 12 | Códigos e zeros à esquerda |
| **13** | **Chaves normalizadas para join** |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| 17 | Freshness |
| 18 | Análises específicas |
| **19** | **EXTRAÇÃO COMPLETA** |
| 20 | Resumo do cenário |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 14.


## 1. Metadados da tabela

In [ ]:
DESCRIBE EXTENDED dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao;

In [ ]:
-- Formato, tamanho e particoes (falha se nao for Delta)
DESCRIBE DETAIL dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao;

In [ ]:
-- Ultimas gravacoes (falha se for view)
DESCRIBE HISTORY dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao LIMIT 20;

## 2. Volumetria e representatividade

Quanto o centro 4014 representa do total da tabela.

In [ ]:
-- 2. VOLUMETRIA DO CENARIO
SELECT
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao)                                   AS linhas_tabela_toda,
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014')                               AS linhas_centro_4014,
  ROUND(100.0 * (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014')
              / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao), 4)                  AS pct_do_total,
  (SELECT COUNT(DISTINCT `cod_centro`) FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao)                     AS centros_na_tabela;

## 3. Confirmação do filtro

Confirma que o valor `4014` existe e que não há variação de formato
(espaços, zeros à esquerda) que faça o filtro perder linhas silenciosamente.

**Se retornar mais de uma linha, o filtro `= '4014'` está incompleto.**

In [ ]:
-- 3. O FILTRO PEGOU TUDO?
SELECT CAST(`cod_centro` AS STRING)                    AS valor_bruto,
       length(CAST(`cod_centro` AS STRING))            AS comprimento,
       COUNT(*)                                   AS linhas
FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao
WHERE regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '') = '4014'
   OR trim(CAST(`cod_centro` AS STRING)) = '4014'
GROUP BY CAST(`cod_centro` AS STRING), length(CAST(`cod_centro` AS STRING))
ORDER BY linhas DESC;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional
multiplicando as linhas.

In [ ]:
-- 4. GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'),
g AS (
  SELECT 'num_pedido_compra + num_item_pedido_compra' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra` FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_pedido_compra + num_item_pedido_compra + cod_material' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra`, `cod_material` FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_pedido_compra' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_pedido_compra` FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014')
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `num_pedido_compra + num_item_pedido_compra`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [ ]:
-- 5. CHAVES DUPLICADAS
SELECT `num_pedido_compra`, `num_item_pedido_compra`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
GROUP BY `num_pedido_compra`, `num_item_pedido_compra`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [ ]:
-- 5.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH cen AS (
  SELECT * FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
),
dup AS (
  SELECT `num_pedido_compra`, `num_item_pedido_compra` FROM cen GROUP BY `num_pedido_compra`, `num_item_pedido_compra` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM cen c JOIN dup ON c.`num_pedido_compra` <=> dup.`num_pedido_compra` AND c.`num_item_pedido_compra` <=> dup.`num_item_pedido_compra`
),
agg AS (
  SELECT `num_pedido_compra`, `num_item_pedido_compra`,
         COUNT(DISTINCT `cod_material`) AS `cod_material`,
         COUNT(DISTINCT `desc_material`) AS `desc_material`,
         COUNT(DISTINCT `cod_texto_breve`) AS `cod_texto_breve`,
         COUNT(DISTINCT `tp_grupo_mercadorias`) AS `tp_grupo_mercadorias`,
         COUNT(DISTINCT `qt_pedido`) AS `qt_pedido`,
         COUNT(DISTINCT `cod_unidade_medida_pedido`) AS `cod_unidade_medida_pedido`,
         COUNT(DISTINCT `cod_unidade_medida_basica`) AS `cod_unidade_medida_basica`,
         COUNT(DISTINCT `vl_preco_liquido`) AS `vl_preco_liquido`,
         COUNT(DISTINCT `qt_unidade_preco`) AS `qt_unidade_preco`,
         COUNT(DISTINCT `cod_moeda`) AS `cod_moeda`,
         COUNT(DISTINCT `cod_fornecedor`) AS `cod_fornecedor`,
         COUNT(DISTINCT `nm_fornecedor`) AS `nm_fornecedor`,
         COUNT(DISTINCT `st_liberacao`) AS `st_liberacao`,
         COUNT(DISTINCT `cod_liberacao`) AS `cod_liberacao`,
         COUNT(DISTINCT `cod_estrat_liberacao`) AS `cod_estrat_liberacao`,
         COUNT(DISTINCT `cod_organizacao_compras`) AS `cod_organizacao_compras`,
         COUNT(DISTINCT `cod_empresa`) AS `cod_empresa`,
         COUNT(DISTINCT `cod_centro`) AS `cod_centro`,
         COUNT(DISTINCT `cod_grupo_compradores`) AS `cod_grupo_compradores`,
         COUNT(DISTINCT `tp_classificacao_contabil`) AS `tp_classificacao_contabil`,
         COUNT(DISTINCT `tp_documento_compras`) AS `tp_documento_compras`,
         COUNT(DISTINCT `tp_categoria_documento`) AS `tp_categoria_documento`,
         COUNT(DISTINCT `cod_categoria_item`) AS `cod_categoria_item`,
         COUNT(DISTINCT `desc_categoria_item`) AS `desc_categoria_item`,
         COUNT(DISTINCT `cod_deposito`) AS `cod_deposito`,
         COUNT(DISTINCT `cod_centro_fornecedor`) AS `cod_centro_fornecedor`,
         COUNT(DISTINCT `num_contrato_basico`) AS `num_contrato_basico`,
         COUNT(DISTINCT `num_registro_info`) AS `num_registro_info`,
         COUNT(DISTINCT `cod_imposto`) AS `cod_imposto`,
         COUNT(DISTINCT `ind_entrega_concluida`) AS `ind_entrega_concluida`,
         COUNT(DISTINCT `num_acompanhamento`) AS `num_acompanhamento`,
         COUNT(DISTINCT `dt_pedido`) AS `dt_pedido`,
         COUNT(DISTINCT `dt_criacao`) AS `dt_criacao`,
         COUNT(DISTINCT `dt_remessa_item`) AS `dt_remessa_item`,
         COUNT(DISTINCT `dt_remessa_primeira`) AS `dt_remessa_primeira`,
         COUNT(DISTINCT `dt_remessa_ultima`) AS `dt_remessa_ultima`,
         COUNT(DISTINCT `qt_divisoes_remessa`) AS `qt_divisoes_remessa`,
         COUNT(DISTINCT `cod_requisicao_compra`) AS `cod_requisicao_compra`,
         COUNT(DISTINCT `cod_item_requisicao_compra`) AS `cod_item_requisicao_compra`,
         COUNT(DISTINCT `vl_requisicao_compra`) AS `vl_requisicao_compra`,
         COUNT(DISTINCT `qt_unidade_preco_requisicao`) AS `qt_unidade_preco_requisicao`,
         COUNT(DISTINCT `qt_solicitada`) AS `qt_solicitada`,
         COUNT(DISTINCT `dateingest`) AS `dateingest`,
         COUNT(DISTINCT `yearingest`) AS `yearingest`,
         COUNT(DISTINCT `monthingest`) AS `monthingest`
  FROM d GROUP BY `num_pedido_compra`, `num_item_pedido_compra`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(45,
    'cod_material', MAX(`cod_material`),
    'desc_material', MAX(`desc_material`),
    'cod_texto_breve', MAX(`cod_texto_breve`),
    'tp_grupo_mercadorias', MAX(`tp_grupo_mercadorias`),
    'qt_pedido', MAX(`qt_pedido`),
    'cod_unidade_medida_pedido', MAX(`cod_unidade_medida_pedido`),
    'cod_unidade_medida_basica', MAX(`cod_unidade_medida_basica`),
    'vl_preco_liquido', MAX(`vl_preco_liquido`),
    'qt_unidade_preco', MAX(`qt_unidade_preco`),
    'cod_moeda', MAX(`cod_moeda`),
    'cod_fornecedor', MAX(`cod_fornecedor`),
    'nm_fornecedor', MAX(`nm_fornecedor`),
    'st_liberacao', MAX(`st_liberacao`),
    'cod_liberacao', MAX(`cod_liberacao`),
    'cod_estrat_liberacao', MAX(`cod_estrat_liberacao`),
    'cod_organizacao_compras', MAX(`cod_organizacao_compras`),
    'cod_empresa', MAX(`cod_empresa`),
    'cod_centro', MAX(`cod_centro`),
    'cod_grupo_compradores', MAX(`cod_grupo_compradores`),
    'tp_classificacao_contabil', MAX(`tp_classificacao_contabil`),
    'tp_documento_compras', MAX(`tp_documento_compras`),
    'tp_categoria_documento', MAX(`tp_categoria_documento`),
    'cod_categoria_item', MAX(`cod_categoria_item`),
    'desc_categoria_item', MAX(`desc_categoria_item`),
    'cod_deposito', MAX(`cod_deposito`),
    'cod_centro_fornecedor', MAX(`cod_centro_fornecedor`),
    'num_contrato_basico', MAX(`num_contrato_basico`),
    'num_registro_info', MAX(`num_registro_info`),
    'cod_imposto', MAX(`cod_imposto`),
    'ind_entrega_concluida', MAX(`ind_entrega_concluida`),
    'num_acompanhamento', MAX(`num_acompanhamento`),
    'dt_pedido', MAX(`dt_pedido`),
    'dt_criacao', MAX(`dt_criacao`),
    'dt_remessa_item', MAX(`dt_remessa_item`),
    'dt_remessa_primeira', MAX(`dt_remessa_primeira`),
    'dt_remessa_ultima', MAX(`dt_remessa_ultima`),
    'qt_divisoes_remessa', MAX(`qt_divisoes_remessa`),
    'cod_requisicao_compra', MAX(`cod_requisicao_compra`),
    'cod_item_requisicao_compra', MAX(`cod_item_requisicao_compra`),
    'vl_requisicao_compra', MAX(`vl_requisicao_compra`),
    'qt_unidade_preco_requisicao', MAX(`qt_unidade_preco_requisicao`),
    'qt_solicitada', MAX(`qt_solicitada`),
    'dateingest', MAX(`dateingest`),
    'yearingest', MAX(`yearingest`),
    'monthingest', MAX(`monthingest`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada **neste centro**.

Uma coluna pode ter dado na tabela toda e estar vazia no centro 4014 — ou o contrário.
Por isso a varredura é feita sobre o recorte, não sobre a base completa.

In [ ]:
-- 6. PREENCHIMENTO NO CENTRO 4014
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'),
perf AS (
  SELECT stack(47,
    'num_pedido_compra', 'bigint', COUNT_IF(`num_pedido_compra` IS NULL), 0L, COUNT_IF(`num_pedido_compra` = 0),
    'num_item_pedido_compra', 'bigint', COUNT_IF(`num_item_pedido_compra` IS NULL), 0L, COUNT_IF(`num_item_pedido_compra` = 0),
    'cod_material', 'bigint', COUNT_IF(`cod_material` IS NULL), 0L, COUNT_IF(`cod_material` = 0),
    'desc_material', 'string', COUNT_IF(`desc_material` IS NULL), COUNT_IF(`desc_material` IS NOT NULL AND lower(trim(`desc_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_texto_breve', 'string', COUNT_IF(`cod_texto_breve` IS NULL), COUNT_IF(`cod_texto_breve` IS NOT NULL AND lower(trim(`cod_texto_breve`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_texto_breve`) RLIKE '^0+([.,]0+)?$'),
    'tp_grupo_mercadorias', 'string', COUNT_IF(`tp_grupo_mercadorias` IS NULL), COUNT_IF(`tp_grupo_mercadorias` IS NOT NULL AND lower(trim(`tp_grupo_mercadorias`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_grupo_mercadorias`) RLIKE '^0+([.,]0+)?$'),
    'qt_pedido', 'decimal(13,3)', COUNT_IF(`qt_pedido` IS NULL), 0L, COUNT_IF(`qt_pedido` = 0),
    'cod_unidade_medida_pedido', 'string', COUNT_IF(`cod_unidade_medida_pedido` IS NULL), COUNT_IF(`cod_unidade_medida_pedido` IS NOT NULL AND lower(trim(`cod_unidade_medida_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_unidade_medida_pedido`) RLIKE '^0+([.,]0+)?$'),
    'cod_unidade_medida_basica', 'string', COUNT_IF(`cod_unidade_medida_basica` IS NULL), COUNT_IF(`cod_unidade_medida_basica` IS NOT NULL AND lower(trim(`cod_unidade_medida_basica`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_unidade_medida_basica`) RLIKE '^0+([.,]0+)?$'),
    'vl_preco_liquido', 'double', COUNT_IF(`vl_preco_liquido` IS NULL), 0L, COUNT_IF(`vl_preco_liquido` = 0),
    'qt_unidade_preco', 'decimal(5,0)', COUNT_IF(`qt_unidade_preco` IS NULL), 0L, COUNT_IF(`qt_unidade_preco` = 0),
    'cod_moeda', 'string', COUNT_IF(`cod_moeda` IS NULL), COUNT_IF(`cod_moeda` IS NOT NULL AND lower(trim(`cod_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda`) RLIKE '^0+([.,]0+)?$'),
    'cod_fornecedor', 'bigint', COUNT_IF(`cod_fornecedor` IS NULL), 0L, COUNT_IF(`cod_fornecedor` = 0),
    'nm_fornecedor', 'string', COUNT_IF(`nm_fornecedor` IS NULL), COUNT_IF(`nm_fornecedor` IS NOT NULL AND lower(trim(`nm_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'st_liberacao', 'string', COUNT_IF(`st_liberacao` IS NULL), COUNT_IF(`st_liberacao` IS NOT NULL AND lower(trim(`st_liberacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_liberacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_liberacao', 'string', COUNT_IF(`cod_liberacao` IS NULL), COUNT_IF(`cod_liberacao` IS NOT NULL AND lower(trim(`cod_liberacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_liberacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_estrat_liberacao', 'string', COUNT_IF(`cod_estrat_liberacao` IS NULL), COUNT_IF(`cod_estrat_liberacao` IS NOT NULL AND lower(trim(`cod_estrat_liberacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_estrat_liberacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_organizacao_compras', 'string', COUNT_IF(`cod_organizacao_compras` IS NULL), COUNT_IF(`cod_organizacao_compras` IS NOT NULL AND lower(trim(`cod_organizacao_compras`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_organizacao_compras`) RLIKE '^0+([.,]0+)?$'),
    'cod_empresa', 'string', COUNT_IF(`cod_empresa` IS NULL), COUNT_IF(`cod_empresa` IS NOT NULL AND lower(trim(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_empresa`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_compradores', 'string', COUNT_IF(`cod_grupo_compradores` IS NULL), COUNT_IF(`cod_grupo_compradores` IS NOT NULL AND lower(trim(`cod_grupo_compradores`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_compradores`) RLIKE '^0+([.,]0+)?$'),
    'tp_classificacao_contabil', 'string', COUNT_IF(`tp_classificacao_contabil` IS NULL), COUNT_IF(`tp_classificacao_contabil` IS NOT NULL AND lower(trim(`tp_classificacao_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_classificacao_contabil`) RLIKE '^0+([.,]0+)?$'),
    'tp_documento_compras', 'string', COUNT_IF(`tp_documento_compras` IS NULL), COUNT_IF(`tp_documento_compras` IS NOT NULL AND lower(trim(`tp_documento_compras`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_documento_compras`) RLIKE '^0+([.,]0+)?$'),
    'tp_categoria_documento', 'string', COUNT_IF(`tp_categoria_documento` IS NULL), COUNT_IF(`tp_categoria_documento` IS NOT NULL AND lower(trim(`tp_categoria_documento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_categoria_documento`) RLIKE '^0+([.,]0+)?$'),
    'cod_categoria_item', 'string', COUNT_IF(`cod_categoria_item` IS NULL), COUNT_IF(`cod_categoria_item` IS NOT NULL AND lower(trim(`cod_categoria_item`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_categoria_item`) RLIKE '^0+([.,]0+)?$'),
    'desc_categoria_item', 'string', COUNT_IF(`desc_categoria_item` IS NULL), COUNT_IF(`desc_categoria_item` IS NOT NULL AND lower(trim(`desc_categoria_item`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_categoria_item`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito', 'string', COUNT_IF(`cod_deposito` IS NULL), COUNT_IF(`cod_deposito` IS NOT NULL AND lower(trim(`cod_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_fornecedor', 'bigint', COUNT_IF(`cod_centro_fornecedor` IS NULL), 0L, COUNT_IF(`cod_centro_fornecedor` = 0),
    'num_contrato_basico', 'bigint', COUNT_IF(`num_contrato_basico` IS NULL), 0L, COUNT_IF(`num_contrato_basico` = 0),
    'num_registro_info', 'bigint', COUNT_IF(`num_registro_info` IS NULL), 0L, COUNT_IF(`num_registro_info` = 0),
    'cod_imposto', 'string', COUNT_IF(`cod_imposto` IS NULL), COUNT_IF(`cod_imposto` IS NOT NULL AND lower(trim(`cod_imposto`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_imposto`) RLIKE '^0+([.,]0+)?$'),
    'ind_entrega_concluida', 'string', COUNT_IF(`ind_entrega_concluida` IS NULL), COUNT_IF(`ind_entrega_concluida` IS NOT NULL AND lower(trim(`ind_entrega_concluida`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_entrega_concluida`) RLIKE '^0+([.,]0+)?$'),
    'num_acompanhamento', 'string', COUNT_IF(`num_acompanhamento` IS NULL), COUNT_IF(`num_acompanhamento` IS NOT NULL AND lower(trim(`num_acompanhamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_acompanhamento`) RLIKE '^0+([.,]0+)?$'),
    'dt_pedido', 'date', COUNT_IF(`dt_pedido` IS NULL), 0L, 0L,
    'dt_criacao', 'date', COUNT_IF(`dt_criacao` IS NULL), 0L, 0L,
    'dt_remessa_item', 'date', COUNT_IF(`dt_remessa_item` IS NULL), 0L, 0L,
    'dt_remessa_primeira', 'date', COUNT_IF(`dt_remessa_primeira` IS NULL), 0L, 0L,
    'dt_remessa_ultima', 'date', COUNT_IF(`dt_remessa_ultima` IS NULL), 0L, 0L,
    'qt_divisoes_remessa', 'bigint', COUNT_IF(`qt_divisoes_remessa` IS NULL), 0L, COUNT_IF(`qt_divisoes_remessa` = 0),
    'cod_requisicao_compra', 'bigint', COUNT_IF(`cod_requisicao_compra` IS NULL), 0L, COUNT_IF(`cod_requisicao_compra` = 0),
    'cod_item_requisicao_compra', 'bigint', COUNT_IF(`cod_item_requisicao_compra` IS NULL), 0L, COUNT_IF(`cod_item_requisicao_compra` = 0),
    'vl_requisicao_compra', 'double', COUNT_IF(`vl_requisicao_compra` IS NULL), 0L, COUNT_IF(`vl_requisicao_compra` = 0),
    'qt_unidade_preco_requisicao', 'decimal(5,0)', COUNT_IF(`qt_unidade_preco_requisicao` IS NULL), 0L, COUNT_IF(`qt_unidade_preco_requisicao` = 0),
    'qt_solicitada', 'decimal(13,3)', COUNT_IF(`qt_solicitada` IS NULL), 0L, COUNT_IF(`qt_solicitada` = 0),
    'dateingest', 'date', COUNT_IF(`dateingest` IS NULL), 0L, 0L,
    'yearingest', 'string', COUNT_IF(`yearingest` IS NULL), COUNT_IF(`yearingest` IS NOT NULL AND lower(trim(`yearingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`yearingest`) RLIKE '^0+([.,]0+)?$'),
    'monthingest', 'string', COUNT_IF(`monthingest` IS NULL), COUNT_IF(`monthingest` IS NOT NULL AND lower(trim(`monthingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`monthingest`) RLIKE '^0+([.,]0+)?$')
  ) AS (coluna, tipo, nulos, vazios, zeros)
  FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros,
       t.total - p.nulos - p.vazios - p.zeros                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                       THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                              AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade no cenário

In [ ]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'),
card AS (
  SELECT stack(47,
    'num_pedido_compra', 'bigint', approx_count_distinct(`num_pedido_compra`),
    'num_item_pedido_compra', 'bigint', approx_count_distinct(`num_item_pedido_compra`),
    'cod_material', 'bigint', approx_count_distinct(`cod_material`),
    'desc_material', 'string', approx_count_distinct(`desc_material`),
    'cod_texto_breve', 'string', approx_count_distinct(`cod_texto_breve`),
    'tp_grupo_mercadorias', 'string', approx_count_distinct(`tp_grupo_mercadorias`),
    'qt_pedido', 'decimal(13,3)', approx_count_distinct(`qt_pedido`),
    'cod_unidade_medida_pedido', 'string', approx_count_distinct(`cod_unidade_medida_pedido`),
    'cod_unidade_medida_basica', 'string', approx_count_distinct(`cod_unidade_medida_basica`),
    'vl_preco_liquido', 'double', approx_count_distinct(`vl_preco_liquido`),
    'qt_unidade_preco', 'decimal(5,0)', approx_count_distinct(`qt_unidade_preco`),
    'cod_moeda', 'string', approx_count_distinct(`cod_moeda`),
    'cod_fornecedor', 'bigint', approx_count_distinct(`cod_fornecedor`),
    'nm_fornecedor', 'string', approx_count_distinct(`nm_fornecedor`),
    'st_liberacao', 'string', approx_count_distinct(`st_liberacao`),
    'cod_liberacao', 'string', approx_count_distinct(`cod_liberacao`),
    'cod_estrat_liberacao', 'string', approx_count_distinct(`cod_estrat_liberacao`),
    'cod_organizacao_compras', 'string', approx_count_distinct(`cod_organizacao_compras`),
    'cod_empresa', 'string', approx_count_distinct(`cod_empresa`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_grupo_compradores', 'string', approx_count_distinct(`cod_grupo_compradores`),
    'tp_classificacao_contabil', 'string', approx_count_distinct(`tp_classificacao_contabil`),
    'tp_documento_compras', 'string', approx_count_distinct(`tp_documento_compras`),
    'tp_categoria_documento', 'string', approx_count_distinct(`tp_categoria_documento`),
    'cod_categoria_item', 'string', approx_count_distinct(`cod_categoria_item`),
    'desc_categoria_item', 'string', approx_count_distinct(`desc_categoria_item`),
    'cod_deposito', 'string', approx_count_distinct(`cod_deposito`),
    'cod_centro_fornecedor', 'bigint', approx_count_distinct(`cod_centro_fornecedor`),
    'num_contrato_basico', 'bigint', approx_count_distinct(`num_contrato_basico`),
    'num_registro_info', 'bigint', approx_count_distinct(`num_registro_info`),
    'cod_imposto', 'string', approx_count_distinct(`cod_imposto`),
    'ind_entrega_concluida', 'string', approx_count_distinct(`ind_entrega_concluida`),
    'num_acompanhamento', 'string', approx_count_distinct(`num_acompanhamento`),
    'dt_pedido', 'date', approx_count_distinct(`dt_pedido`),
    'dt_criacao', 'date', approx_count_distinct(`dt_criacao`),
    'dt_remessa_item', 'date', approx_count_distinct(`dt_remessa_item`),
    'dt_remessa_primeira', 'date', approx_count_distinct(`dt_remessa_primeira`),
    'dt_remessa_ultima', 'date', approx_count_distinct(`dt_remessa_ultima`),
    'qt_divisoes_remessa', 'bigint', approx_count_distinct(`qt_divisoes_remessa`),
    'cod_requisicao_compra', 'bigint', approx_count_distinct(`cod_requisicao_compra`),
    'cod_item_requisicao_compra', 'bigint', approx_count_distinct(`cod_item_requisicao_compra`),
    'vl_requisicao_compra', 'double', approx_count_distinct(`vl_requisicao_compra`),
    'qt_unidade_preco_requisicao', 'decimal(5,0)', approx_count_distinct(`qt_unidade_preco_requisicao`),
    'qt_solicitada', 'decimal(13,3)', approx_count_distinct(`qt_solicitada`),
    'dateingest', 'date', approx_count_distinct(`dateingest`),
    'yearingest', 'string', approx_count_distinct(`yearingest`),
    'monthingest', 'string', approx_count_distinct(`monthingest`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma, dentro do cenário.

In [ ]:
-- 8. DOMINIO DAS CATEGORICAS
(SELECT 'tp_grupo_mercadorias' AS coluna, CAST(`tp_grupo_mercadorias` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014' GROUP BY `tp_grupo_mercadorias` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_moeda' AS coluna, CAST(`cod_moeda` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014' GROUP BY `cod_moeda` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_liberacao' AS coluna, CAST(`st_liberacao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014' GROUP BY `st_liberacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_organizacao_compras' AS coluna, CAST(`cod_organizacao_compras` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014' GROUP BY `cod_organizacao_compras` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014' GROUP BY `cod_empresa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_grupo_compradores' AS coluna, CAST(`cod_grupo_compradores` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014' GROUP BY `cod_grupo_compradores` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_classificacao_contabil' AS coluna, CAST(`tp_classificacao_contabil` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014' GROUP BY `tp_classificacao_contabil` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_documento_compras' AS coluna, CAST(`tp_documento_compras` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014' GROUP BY `tp_documento_compras` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_categoria_documento' AS coluna, CAST(`tp_categoria_documento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014' GROUP BY `tp_categoria_documento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_categoria_item' AS coluna, CAST(`cod_categoria_item` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014' GROUP BY `cod_categoria_item` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'desc_categoria_item' AS coluna, CAST(`desc_categoria_item` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014' GROUP BY `desc_categoria_item` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_deposito' AS coluna, CAST(`cod_deposito` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014' GROUP BY `cod_deposito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_entrega_concluida' AS coluna, CAST(`ind_entrega_concluida` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014' GROUP BY `ind_entrega_concluida` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Perfil dos campos numéricos

Campos `double` exigem tolerância de 0,005 na comparação com o SAP.

In [ ]:
-- 9. PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(6,
    'qt_pedido', 'decimal(13,3)', COUNT(`qt_pedido`), CAST(MIN(`qt_pedido`) AS DOUBLE), CAST(MAX(`qt_pedido`) AS DOUBLE), CAST(AVG(`qt_pedido`) AS DOUBLE), CAST(percentile_approx(`qt_pedido`, 0.5) AS DOUBLE), COUNT_IF(`qt_pedido` < 0), COUNT_IF(`qt_pedido` = 0),
    'vl_preco_liquido', 'double', COUNT(`vl_preco_liquido`), CAST(MIN(`vl_preco_liquido`) AS DOUBLE), CAST(MAX(`vl_preco_liquido`) AS DOUBLE), CAST(AVG(`vl_preco_liquido`) AS DOUBLE), CAST(percentile_approx(`vl_preco_liquido`, 0.5) AS DOUBLE), COUNT_IF(`vl_preco_liquido` < 0), COUNT_IF(`vl_preco_liquido` = 0),
    'qt_unidade_preco', 'decimal(5,0)', COUNT(`qt_unidade_preco`), CAST(MIN(`qt_unidade_preco`) AS DOUBLE), CAST(MAX(`qt_unidade_preco`) AS DOUBLE), CAST(AVG(`qt_unidade_preco`) AS DOUBLE), CAST(percentile_approx(`qt_unidade_preco`, 0.5) AS DOUBLE), COUNT_IF(`qt_unidade_preco` < 0), COUNT_IF(`qt_unidade_preco` = 0),
    'qt_divisoes_remessa', 'bigint', COUNT(`qt_divisoes_remessa`), CAST(MIN(`qt_divisoes_remessa`) AS DOUBLE), CAST(MAX(`qt_divisoes_remessa`) AS DOUBLE), CAST(AVG(`qt_divisoes_remessa`) AS DOUBLE), CAST(percentile_approx(`qt_divisoes_remessa`, 0.5) AS DOUBLE), COUNT_IF(`qt_divisoes_remessa` < 0), COUNT_IF(`qt_divisoes_remessa` = 0),
    'vl_requisicao_compra', 'double', COUNT(`vl_requisicao_compra`), CAST(MIN(`vl_requisicao_compra`) AS DOUBLE), CAST(MAX(`vl_requisicao_compra`) AS DOUBLE), CAST(AVG(`vl_requisicao_compra`) AS DOUBLE), CAST(percentile_approx(`vl_requisicao_compra`, 0.5) AS DOUBLE), COUNT_IF(`vl_requisicao_compra` < 0), COUNT_IF(`vl_requisicao_compra` = 0),
    'qt_solicitada', 'decimal(13,3)', COUNT(`qt_solicitada`), CAST(MIN(`qt_solicitada`) AS DOUBLE), CAST(MAX(`qt_solicitada`) AS DOUBLE), CAST(AVG(`qt_solicitada`) AS DOUBLE), CAST(percentile_approx(`qt_solicitada`, 0.5) AS DOUBLE), COUNT_IF(`qt_solicitada` < 0), COUNT_IF(`qt_solicitada` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
)
ORDER BY coluna;

## 10. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel extraído do SAP e compare linha a linha.
Divergência de total é o teste mais rápido para detectar registro faltando ou duplicado —
e cobre o ponto cego da contagem de linhas, que sozinha não prova nada.

In [ ]:
-- 10. TOTAIS PARA CONCILIACAO
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(6,
    'qt_pedido', CAST(SUM(`qt_pedido`) AS DOUBLE), CAST(ROUND(SUM(`qt_pedido`), 2) AS STRING), COUNT(`qt_pedido`),
    'vl_preco_liquido', CAST(SUM(`vl_preco_liquido`) AS DOUBLE), CAST(ROUND(SUM(`vl_preco_liquido`), 2) AS STRING), COUNT(`vl_preco_liquido`),
    'qt_unidade_preco', CAST(SUM(`qt_unidade_preco`) AS DOUBLE), CAST(ROUND(SUM(`qt_unidade_preco`), 2) AS STRING), COUNT(`qt_unidade_preco`),
    'qt_divisoes_remessa', CAST(SUM(`qt_divisoes_remessa`) AS DOUBLE), CAST(ROUND(SUM(`qt_divisoes_remessa`), 2) AS STRING), COUNT(`qt_divisoes_remessa`),
    'vl_requisicao_compra', CAST(SUM(`vl_requisicao_compra`) AS DOUBLE), CAST(ROUND(SUM(`vl_requisicao_compra`), 2) AS STRING), COUNT(`vl_requisicao_compra`),
    'qt_solicitada', CAST(SUM(`qt_solicitada`) AS DOUBLE), CAST(ROUND(SUM(`qt_solicitada`), 2) AS STRING), COUNT(`qt_solicitada`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
)
ORDER BY coluna;

## 11.1 Datas em tipo nativo

In [ ]:
-- 11.1 DATAS NATIVAS
SELECT * FROM (
  SELECT stack(6,
    'dt_pedido', COUNT_IF(`dt_pedido` IS NULL), CAST(MIN(`dt_pedido`) AS STRING), CAST(MAX(`dt_pedido`) AS STRING), COUNT(DISTINCT `dt_pedido`), COUNT_IF(`dt_pedido` > current_date()),
    'dt_criacao', COUNT_IF(`dt_criacao` IS NULL), CAST(MIN(`dt_criacao`) AS STRING), CAST(MAX(`dt_criacao`) AS STRING), COUNT(DISTINCT `dt_criacao`), COUNT_IF(`dt_criacao` > current_date()),
    'dt_remessa_item', COUNT_IF(`dt_remessa_item` IS NULL), CAST(MIN(`dt_remessa_item`) AS STRING), CAST(MAX(`dt_remessa_item`) AS STRING), COUNT(DISTINCT `dt_remessa_item`), COUNT_IF(`dt_remessa_item` > current_date()),
    'dt_remessa_primeira', COUNT_IF(`dt_remessa_primeira` IS NULL), CAST(MIN(`dt_remessa_primeira`) AS STRING), CAST(MAX(`dt_remessa_primeira`) AS STRING), COUNT(DISTINCT `dt_remessa_primeira`), COUNT_IF(`dt_remessa_primeira` > current_date()),
    'dt_remessa_ultima', COUNT_IF(`dt_remessa_ultima` IS NULL), CAST(MIN(`dt_remessa_ultima`) AS STRING), CAST(MAX(`dt_remessa_ultima`) AS STRING), COUNT(DISTINCT `dt_remessa_ultima`), COUNT_IF(`dt_remessa_ultima` > current_date()),
    'dateingest', COUNT_IF(`dateingest` IS NULL), CAST(MIN(`dateingest`) AS STRING), CAST(MAX(`dateingest`) AS STRING), COUNT(DISTINCT `dateingest`), COUNT_IF(`dateingest` > current_date())
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras)
  FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
)
ORDER BY coluna;

## 12. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

In [ ]:
-- 12. CODIGOS
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(7,
    'num_pedido_compra', 'bigint', COUNT_IF(CAST(`num_pedido_compra` AS STRING) IS NULL OR trim(CAST(`num_pedido_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_pedido_compra` AS STRING)))), MAX(length(trim(CAST(`num_pedido_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_pedido_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_pedido_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_pedido_compra` AS STRING)), '^0+', '')),
    'num_item_pedido_compra', 'bigint', COUNT_IF(CAST(`num_item_pedido_compra` AS STRING) IS NULL OR trim(CAST(`num_item_pedido_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_pedido_compra` AS STRING)))), MAX(length(trim(CAST(`num_item_pedido_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_item_pedido_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_pedido_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_pedido_compra` AS STRING)), '^0+', '')),
    'cod_material', 'bigint', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_fornecedor', 'bigint', COUNT_IF(CAST(`cod_fornecedor` AS STRING) IS NULL OR trim(CAST(`cod_fornecedor` AS STRING)) = ''), MIN(length(trim(CAST(`cod_fornecedor` AS STRING)))), MAX(length(trim(CAST(`cod_fornecedor` AS STRING)))), COUNT_IF(trim(CAST(`cod_fornecedor` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_fornecedor` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_fornecedor` AS STRING)), '^0+', '')),
    'num_contrato_basico', 'bigint', COUNT_IF(CAST(`num_contrato_basico` AS STRING) IS NULL OR trim(CAST(`num_contrato_basico` AS STRING)) = ''), MIN(length(trim(CAST(`num_contrato_basico` AS STRING)))), MAX(length(trim(CAST(`num_contrato_basico` AS STRING)))), COUNT_IF(trim(CAST(`num_contrato_basico` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_contrato_basico` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_contrato_basico` AS STRING)), '^0+', '')),
    'num_registro_info', 'bigint', COUNT_IF(CAST(`num_registro_info` AS STRING) IS NULL OR trim(CAST(`num_registro_info` AS STRING)) = ''), MIN(length(trim(CAST(`num_registro_info` AS STRING)))), MAX(length(trim(CAST(`num_registro_info` AS STRING)))), COUNT_IF(trim(CAST(`num_registro_info` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_registro_info` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_registro_info` AS STRING)), '^0+', '')),
    'cod_requisicao_compra', 'bigint', COUNT_IF(CAST(`cod_requisicao_compra` AS STRING) IS NULL OR trim(CAST(`cod_requisicao_compra` AS STRING)) = ''), MIN(length(trim(CAST(`cod_requisicao_compra` AS STRING)))), MAX(length(trim(CAST(`cod_requisicao_compra` AS STRING)))), COUNT_IF(trim(CAST(`cod_requisicao_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_requisicao_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_requisicao_compra` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
)
ORDER BY coluna;

## 13. Chaves normalizadas para join com o SAP

Lista das chaves já **sem zeros à esquerda**, prontas para colar no Excel
e cruzar com o extrato do SAP via PROCV/ÍNDICE.

Baixe como CSV e use para identificar registros presentes de um lado e ausentes do outro.

In [ ]:
-- 13. CHAVES NORMALIZADAS (para cruzar com o SAP)
SELECT DISTINCT
       regexp_replace(trim(CAST(`num_pedido_compra` AS STRING)), '^0+', '') AS `num_pedido_compra_norm`,
       regexp_replace(trim(CAST(`num_item_pedido_compra` AS STRING)), '^0+', '') AS `num_item_pedido_compra_norm`
FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
ORDER BY 1, 2;

## 14. Checksum de linha

Gera uma impressão digital de cada linha. Dois usos:

- **Contar linhas realmente distintas** — se `linhas` for maior que `linhas_unicas`,
  existem registros 100% idênticos (duplicata real)
- **Comparação rápida** — aplicando a mesma concatenação no SAP, dá para achar
  divergências sem comparar campo a campo

In [ ]:
-- 14. CHECKSUM DE LINHA
SELECT COUNT(*)                                                 AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`cod_texto_breve` AS STRING), ''), COALESCE(CAST(`tp_grupo_mercadorias` AS STRING), ''), COALESCE(CAST(`qt_pedido` AS STRING), ''), COALESCE(CAST(`cod_unidade_medida_pedido` AS STRING), ''), COALESCE(CAST(`cod_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`vl_preco_liquido` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`st_liberacao` AS STRING), ''), COALESCE(CAST(`cod_liberacao` AS STRING), ''), COALESCE(CAST(`cod_estrat_liberacao` AS STRING), ''), COALESCE(CAST(`cod_organizacao_compras` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`tp_classificacao_contabil` AS STRING), ''), COALESCE(CAST(`tp_documento_compras` AS STRING), ''), COALESCE(CAST(`tp_categoria_documento` AS STRING), ''), COALESCE(CAST(`cod_categoria_item` AS STRING), ''), COALESCE(CAST(`desc_categoria_item` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`cod_centro_fornecedor` AS STRING), ''), COALESCE(CAST(`num_contrato_basico` AS STRING), ''), COALESCE(CAST(`num_registro_info` AS STRING), ''), COALESCE(CAST(`cod_imposto` AS STRING), ''), COALESCE(CAST(`ind_entrega_concluida` AS STRING), ''), COALESCE(CAST(`num_acompanhamento` AS STRING), ''), COALESCE(CAST(`dt_pedido` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`dt_remessa_item` AS STRING), ''), COALESCE(CAST(`dt_remessa_primeira` AS STRING), ''), COALESCE(CAST(`dt_remessa_ultima` AS STRING), ''), COALESCE(CAST(`qt_divisoes_remessa` AS STRING), ''), COALESCE(CAST(`cod_requisicao_compra` AS STRING), ''), COALESCE(CAST(`cod_item_requisicao_compra` AS STRING), ''), COALESCE(CAST(`vl_requisicao_compra` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco_requisicao` AS STRING), ''), COALESCE(CAST(`qt_solicitada` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), ''))))             AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`cod_texto_breve` AS STRING), ''), COALESCE(CAST(`tp_grupo_mercadorias` AS STRING), ''), COALESCE(CAST(`qt_pedido` AS STRING), ''), COALESCE(CAST(`cod_unidade_medida_pedido` AS STRING), ''), COALESCE(CAST(`cod_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`vl_preco_liquido` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`st_liberacao` AS STRING), ''), COALESCE(CAST(`cod_liberacao` AS STRING), ''), COALESCE(CAST(`cod_estrat_liberacao` AS STRING), ''), COALESCE(CAST(`cod_organizacao_compras` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`tp_classificacao_contabil` AS STRING), ''), COALESCE(CAST(`tp_documento_compras` AS STRING), ''), COALESCE(CAST(`tp_categoria_documento` AS STRING), ''), COALESCE(CAST(`cod_categoria_item` AS STRING), ''), COALESCE(CAST(`desc_categoria_item` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`cod_centro_fornecedor` AS STRING), ''), COALESCE(CAST(`num_contrato_basico` AS STRING), ''), COALESCE(CAST(`num_registro_info` AS STRING), ''), COALESCE(CAST(`cod_imposto` AS STRING), ''), COALESCE(CAST(`ind_entrega_concluida` AS STRING), ''), COALESCE(CAST(`num_acompanhamento` AS STRING), ''), COALESCE(CAST(`dt_pedido` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`dt_remessa_item` AS STRING), ''), COALESCE(CAST(`dt_remessa_primeira` AS STRING), ''), COALESCE(CAST(`dt_remessa_ultima` AS STRING), ''), COALESCE(CAST(`qt_divisoes_remessa` AS STRING), ''), COALESCE(CAST(`cod_requisicao_compra` AS STRING), ''), COALESCE(CAST(`cod_item_requisicao_compra` AS STRING), ''), COALESCE(CAST(`vl_requisicao_compra` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco_requisicao` AS STRING), ''), COALESCE(CAST(`qt_solicitada` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), ''))))  AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`cod_texto_breve` AS STRING), ''), COALESCE(CAST(`tp_grupo_mercadorias` AS STRING), ''), COALESCE(CAST(`qt_pedido` AS STRING), ''), COALESCE(CAST(`cod_unidade_medida_pedido` AS STRING), ''), COALESCE(CAST(`cod_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`vl_preco_liquido` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`st_liberacao` AS STRING), ''), COALESCE(CAST(`cod_liberacao` AS STRING), ''), COALESCE(CAST(`cod_estrat_liberacao` AS STRING), ''), COALESCE(CAST(`cod_organizacao_compras` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`tp_classificacao_contabil` AS STRING), ''), COALESCE(CAST(`tp_documento_compras` AS STRING), ''), COALESCE(CAST(`tp_categoria_documento` AS STRING), ''), COALESCE(CAST(`cod_categoria_item` AS STRING), ''), COALESCE(CAST(`desc_categoria_item` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`cod_centro_fornecedor` AS STRING), ''), COALESCE(CAST(`num_contrato_basico` AS STRING), ''), COALESCE(CAST(`num_registro_info` AS STRING), ''), COALESCE(CAST(`cod_imposto` AS STRING), ''), COALESCE(CAST(`ind_entrega_concluida` AS STRING), ''), COALESCE(CAST(`num_acompanhamento` AS STRING), ''), COALESCE(CAST(`dt_pedido` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`dt_remessa_item` AS STRING), ''), COALESCE(CAST(`dt_remessa_primeira` AS STRING), ''), COALESCE(CAST(`dt_remessa_ultima` AS STRING), ''), COALESCE(CAST(`qt_divisoes_remessa` AS STRING), ''), COALESCE(CAST(`cod_requisicao_compra` AS STRING), ''), COALESCE(CAST(`cod_item_requisicao_compra` AS STRING), ''), COALESCE(CAST(`vl_requisicao_compra` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco_requisicao` AS STRING), ''), COALESCE(CAST(`qt_solicitada` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014';

## 15. Amostra de linhas completas

In [ ]:
-- 15. AMOSTRA
SELECT * FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
ORDER BY `num_pedido_compra`, `num_item_pedido_compra`
LIMIT 20;

## 16. Distribuição interna do centro 4014

Como o volume se reparte dentro do cenário. Útil para conferir se o extrato do SAP
tem a mesma composição.

In [ ]:
-- 16. DISTRIBUICAO POR cod_grupo_compradores
SELECT COALESCE(NULLIF(trim(CAST(`cod_grupo_compradores` AS STRING)), ''), '(vazio)') AS `cod_grupo_compradores`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_grupo_compradores` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 16. DISTRIBUICAO POR cod_organizacao_compras
SELECT COALESCE(NULLIF(trim(CAST(`cod_organizacao_compras` AS STRING)), ''), '(vazio)') AS `cod_organizacao_compras`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_organizacao_compras` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 16. DISTRIBUICAO POR cod_empresa
SELECT COALESCE(NULLIF(trim(CAST(`cod_empresa` AS STRING)), ''), '(vazio)') AS `cod_empresa`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_empresa` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 16. DISTRIBUICAO POR ind_entrega_concluida
SELECT COALESCE(NULLIF(trim(CAST(`ind_entrega_concluida` AS STRING)), ''), '(vazio)') AS `ind_entrega_concluida`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`ind_entrega_concluida` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

## 17. Freshness

In [ ]:
-- 17. FRESHNESS
SELECT `dateingest` AS data_carga, COUNT(*) AS linhas
FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
GROUP BY `dateingest`
ORDER BY data_carga DESC
LIMIT 30;

## 18. Análises específicas — ME2N

### 18.1 Escopo declarado

O comentário do schema admite que `tp_documento_compras` é sempre `F` — planos de entrega
(categoria L) **não existem** nesta tabela. Ao extrair do SAP, filtre apenas categoria F.

In [ ]:
-- 18.1 TIPO E CATEGORIA DO DOCUMENTO
SELECT tp_documento_compras, tp_categoria_documento,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
GROUP BY tp_documento_compras, tp_categoria_documento
ORDER BY linhas DESC;

### 18.2 Consistência do de-para de categoria de item

In [ ]:
-- 18.2 DE-PARA cod_categoria_item -> desc_categoria_item
SELECT cod_categoria_item,
       COUNT(DISTINCT desc_categoria_item) AS qtd_descricoes,
       CONCAT_WS(' | ', SORT_ARRAY(COLLECT_SET(desc_categoria_item))) AS descricoes,
       COUNT(*) AS linhas,
       CASE WHEN COUNT(DISTINCT desc_categoria_item) > 1
            THEN 'ERRO - codigo com mais de uma descricao' ELSE 'ok' END AS veredito
FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'
GROUP BY cod_categoria_item
ORDER BY qtd_descricoes DESC, linhas DESC;

### 18.3 Itens por pedido e vínculo com requisição

In [ ]:
-- 18.3 ITENS POR PEDIDO
SELECT itens_no_pedido, COUNT(*) AS pedidos
FROM (SELECT num_pedido_compra, COUNT(*) AS itens_no_pedido
        FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014' GROUP BY num_pedido_compra)
GROUP BY itens_no_pedido
ORDER BY itens_no_pedido
LIMIT 30;

In [ ]:
-- 18.3b VINCULO COM REQUISICAO
SELECT COUNT(*) AS linhas,
       COUNT_IF(cod_requisicao_compra IS NOT NULL
            AND trim(CAST(cod_requisicao_compra AS STRING)) <> '') AS com_requisicao,
       COUNT_IF(cod_requisicao_compra IS NULL
             OR trim(CAST(cod_requisicao_compra AS STRING)) = '') AS sem_requisicao,
       COUNT_IF(qt_pedido > qt_solicitada) AS pedido_maior_que_solicitado
FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014';

## 19. EXTRAÇÃO COMPLETA — centro 4014

**Esta é a célula que você baixa para comparar com o SAP.**

Após executar, use **Download → CSV** no resultado.

> **Limites do Databricks:** a tela mostra até 10.000 linhas, mas o download em CSV
> vai além disso. Se o volume for muito grande, use a célula 19.1.

In [ ]:
-- 19. EXTRACAO COMPLETA DO CENARIO
SELECT *
FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao
WHERE `cod_centro` = '4014'
ORDER BY `num_pedido_compra`, `num_item_pedido_compra`;

### 19.1 Alternativa para volume grande _(opcional)_

Descomente para gravar o resultado numa tabela própria e exportar de lá sem limite de tela.

In [ ]:
-- 19.1 GRAVAR EXTRACAO EM TABELA (opcional)
-- CREATE OR REPLACE TABLE dev_procurement.corp_curated.extracao_me2n_4014 AS
-- SELECT * FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014';
--
-- SELECT COUNT(*) FROM dev_procurement.corp_curated.extracao_me2n_4014;
SELECT 'Descomente as linhas acima se precisar gravar a extracao em tabela' AS instrucao;

## 20. Resumo do cenário

Bloco final. **Copie esta saída** e envie ao agente junto com o notebook.

In [ ]:
-- 20. RESUMO DO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014'),
g AS (
  SELECT 'num_pedido_compra + num_item_pedido_compra' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra` FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_pedido_compra + num_item_pedido_compra + cod_material' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra`, `cod_material` FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_pedido_compra' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_pedido_compra` FROM dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao WHERE `cod_centro` = '4014')
)
SELECT 'CENARIO' AS bloco, 'transacao' AS item, 'ME2N' AS valor
UNION ALL SELECT 'CENARIO', 'tabela', 'dev_procurement.corp_curated.vw_ds_proc_me2n_exibicao'
UNION ALL SELECT 'CENARIO', 'filtro', 'cod_centro = 4014'
UNION ALL SELECT 'CENARIO', 'linhas no cenario', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'CENARIO', 'colunas', '47'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Baixar a **seção 19** em CSV — é a base do centro 4014 no Datalake.
2. Extrair a mesma transação no SAP com o filtro `centro = 4014`, **todas as abas**.
3. Anotar a data e hora das duas extrações.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Antes de comparar

- [ ] Zeros à esquerda normalizados nos dois lados (seção 12)
- [ ] Formato de data normalizado (seção 11)
- [ ] Totais numéricos conferidos (seção 10)
- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP (seção 6)
